In [1]:
from datetime import datetime
import pandas as pd
import networkx as nx
from visualize_ean import plot_ean, draw_ean
from visualize_ean_plotly import plot_ean_plotly, draw_ean_plotly, show_ean
from build_ean import build_ean, add_headway_arcs, propagate, enrich_trip_data_with_boundaries, propagate2
import headway_integration as hi
import numpy as np
from collections import defaultdict
import sys
from pathlib import Path as path

project_root = path(r"C:\Users\LeoC\VSCodes\optimizationVinschgau\AusbauVinschgau")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tools.schematic_map.routing import (build_route,get_signal_nodes_on_route,)
from infra_data.scenarios import load_network_csv, load_headways, get_scenario
from tools.RailML2trip_data.reassign_trip_id import reassign_trip_ids_by_departure


# Import infrastructure and timetable, clean and prepare data

In [2]:
nodesDf = load_network_csv("nodes.csv", "node_id")
edgesDf = load_network_csv("edges.csv", "edge_id")
scenario = get_scenario()
#edgesDf["length"] = (edgesDf["node_to"].map(nodesDf["pk_rel"])- edgesDf["node_from"].map(nodesDf["pk_rel"])).abs()
#trip_data = np.load(r"C:\Users\LeoC\VSCodes\optimizationVinschgau\plottingRailML\230215 FBS Fpl 2026-NB-1443983-3.npy", allow_pickle=True).item()
trip_data = np.load(rf"C:\Users\LeoC\VSCodes\optimizationVinschgau\AusbauVinschgau\tools\RailML2trip_data\trip_data_{scenario}.npy", allow_pickle=True).item()
headway_dict = load_headways()
selected_trips_sorted = np.array(list(trip_data.keys()))

In [3]:
trip_data = {
    train_id: [
        (station, arr_dt, dep_dt, True)
        for station, arr_dt, dep_dt in stops
    ]
    for train_id, stops in trip_data.items()
}

In [4]:
trip_data, ordered_old_ids=reassign_trip_ids_by_departure(trip_data)

In [5]:
routing_results = {}

for stop_col in ["stop_slow", "stop_fast"]:

    stops = nodesDf.index[(nodesDf[stop_col] == 1) & (nodesDf["y"] == 0)].tolist()

    for start, end in zip(stops[:-1], stops[1:]):

        routing_results[(start, end)] = build_route(nodesDf, edgesDf, start, end)
        routing_results[(end, start)] = build_route(nodesDf, edgesDf, end, start)

In [6]:
routes = {}
for trip_id, trip in trip_data.items():
    start, end = trip[0][0], trip[-1][0]
    routes[trip_id] = build_route(nodesDf, edgesDf, start, end)

In [7]:
#add side nodes to trip data

from copy import deepcopy

def add_side_nodes_to_trip_data(trip_data, nodesDf):
    """
    Return a copy of trip_data where terminal nodes are replaced by their
    '_side' counterpart, but only for trains running from Merano to Malles
    (i.e. decreasing pk_rel).

    A stop 'X' is replaced by 'X_side' only if:
      - 'X_side' exists in nodesDf, and
      - the train direction is Merano -> Malles.
    """
    trip_data_sides = deepcopy(trip_data)

    available_nodes = set(nodesDf.index)

    for train_id, stops in trip_data_sides.items():

        # Need at least two stops to infer direction
        if len(stops) < 2:
            continue

        pk0 = nodesDf.loc[stops[0][0], "pk_rel"]
        pk1 = nodesDf.loc[stops[1][0], "pk_rel"]

        # Merano -> Malles corresponds to decreasing pk_rel
        merano_to_malles = pk1 > pk0

        if not merano_to_malles:
            continue

        for i, (node, arr, dep, is_stop) in enumerate(stops):

            side_node = f"{node}_side"

            if side_node in available_nodes:
                stops[i] = (side_node, arr, dep, True)

    return trip_data_sides

In [8]:
trip_data_sides = add_side_nodes_to_trip_data(trip_data, nodesDf)

In [9]:
IG = hi.build_infra_graph(edgesDf)
chains, boundary_nodes = hi.extract_chains(IG, nodesDf)

print("Chains:", chains)
print("Boundary Nodes:", boundary_nodes)
trip_data_enriched = enrich_trip_data_with_boundaries(trip_data_sides, routes, nodesDf, boundary_nodes)

Chains: {('Dev_SPON_02', 'Dev_MAL_05'): ['Dev_SPON_02', 'Pr_I_SPON_52', 'Pa_E_SPON_62', 'PL32', 'Pr_I_SPON_72', 'Pr_E_SPON_02', 'HD_801_02', 'HD_801_04', 'SblSpon54860', 'km86800', 'HD_801_06', 'HD_801_07', 'SLU', 'HD_801_08', 'HD_801_10', 'HD_801_12', 'SblSpon58860', 'Pr_E_MAL_01', 'Pa_E_MAL_61', 'Pr_I_MAL_51', 'Dev_MAL_05'], ('Dev_MAL_05', 'MAL'): ['Dev_MAL_05', 'Pa_I_MAL_31d', 'MAL'], ('Dev_MAL_05', 'MAL_side'): ['Dev_MAL_05', 'Pa_I_MAL_31d_side', 'MAL_side'], ('Dev_TEL_01', 'Dev_01_PLA'): ['Dev_TEL_01', 'Pa_I_TEL_31b_side', 'Pr_I_TEL_51_side', 'Pa_I_TEL_31a_side', 'TEL_side', 'Pa_I_TEL_32_side', 'Dev_TEL_02_side', 'Pr_I_TEL_52_side', 'Pa_E_TEL_62_side', 'Pr_E_TEL_02_side', 'Pr_E_RAB_01_side', 'Pr_I_RAB_51_side', 'PL08_side', 'Pa_I_RAB_31_side', 'RAB_side', 'Pa_RAB_32_side', 'Pr_E_RAB_02_side', 'HD_302_02_side', 'Pr_E_PLA_01_side', 'Dev_01_PLA'], ('Dev_01_PLA', 'Dev_PL11_02'): ['Dev_01_PLA', 'Pa_PLA_31', 'PLA', 'Pa_PLA_32', 'PL09', 'Pr_I_PLA_52', 'Pr_E_PLA_02', 'HD_303_02', 'HD_303_

In [10]:
# Remove opposite-direction headways on double-track chains
headway_dict = deepcopy(headway_dict)

# --------------------------------------------------
# Step 1: Count how many infrastructure edges occupy
# each elementary pk interval
# --------------------------------------------------

pk_values = sorted(nodesDf["pk_rel"].unique())

interval_count = defaultdict(int)

for _, edge in edgesDf.iterrows():

    pk1 = nodesDf.loc[edge["node_from"], "pk_rel"]
    pk2 = nodesDf.loc[edge["node_to"], "pk_rel"]

    a, b = sorted((pk1, pk2))

    for left, right in zip(pk_values[:-1], pk_values[1:]):
        if left >= a and right <= b:
            interval_count[(left, right)] += 1


# --------------------------------------------------
# Step 2: Determine whether each chain is entirely
# on double track
# --------------------------------------------------

double_track_chains = set()

for chain_key, chain_nodes in chains.items():

    is_double = True

    for n1, n2 in zip(chain_nodes[:-1], chain_nodes[1:]):

        pk1 = nodesDf.loc[n1, "pk_rel"]
        pk2 = nodesDf.loc[n2, "pk_rel"]

        a, b = sorted((pk1, pk2))

        for left, right in zip(pk_values[:-1], pk_values[1:]):
            if left >= a and right <= b:
                if interval_count[(left, right)] < 2:
                    is_double = False
                    break

        if not is_double:
            break

    if is_double:
        double_track_chains.add(chain_key)


# --------------------------------------------------
# Step 3: Remove opposite-direction headways
# on double-track chains
# --------------------------------------------------

for key in list(headway_dict):

    chain_key, cat1, cat2 = key

    if chain_key in double_track_chains:

        opposite = (
            ("up" in cat1 and "down" in cat2)
            or
            ("down" in cat1 and "up" in cat2)
        )

        if opposite:
            del headway_dict[key]

# Build EAN, add headway constraints

In [11]:
constraints, skipped = hi.assemble_headway_constraints2(trip_data_sides, trip_data_enriched, routes, nodesDf, chains, headway_dict)
print("Constraints:", constraints)
print("Skipped:", skipped)

G_scheduled = build_ean(trip_data_enriched)
G_scheduled = add_headway_arcs(G_scheduled, constraints)
assert nx.is_directed_acyclic_graph(G_scheduled), "graph must stay a DAG"

[assemble_headway_constraints] 301 entries skipped -- inspect `skipped` for details.
Constraints: [{'train_i': 1, 'seq_i': 14, 'event_i': 'dep', 'train_j': 2, 'seq_j': 8, 'event_j': 'dep', 'min_headway': 197.50191624270101, 'resource': ('Dev_STA_02', 'Dev_PL13_01')}, {'train_i': 2, 'seq_i': 9, 'event_i': 'arr', 'train_j': 22, 'seq_j': 22, 'event_j': 'dep', 'min_headway': 68.15063086506221, 'resource': ('Dev_STA_02', 'Dev_PL13_01')}, {'train_i': 22, 'seq_i': 23, 'event_i': 'arr', 'train_j': 3, 'seq_j': 14, 'event_j': 'dep', 'min_headway': 96.51438269824268, 'resource': ('Dev_STA_02', 'Dev_PL13_01')}, {'train_i': 3, 'seq_i': 15, 'event_i': 'arr', 'train_j': 23, 'seq_j': 16, 'event_j': 'dep', 'min_headway': 68.1306672739332, 'resource': ('Dev_STA_02', 'Dev_PL13_01')}, {'train_i': 23, 'seq_i': 17, 'event_i': 'arr', 'train_j': 4, 'seq_j': 14, 'event_j': 'dep', 'min_headway': 96.52838208017158, 'resource': ('Dev_STA_02', 'Dev_PL13_01')}, {'train_i': 4, 'seq_i': 14, 'event_i': 'dep', 'train_j

In [12]:
bad = []

for c in constraints:
    ti = hi.event_time_sec(c["train_i"], c["seq_i"], c["event_i"], trip_data_enriched)
    tj = hi.event_time_sec(c["train_j"], c["seq_j"], c["event_j"], trip_data_enriched)
    if ti > tj:
        bad.append((c, ti, tj))

print(len(bad))

0


# Stochastic Perturbation

In [18]:
node_perturbations, edge_perturbations = hi.generate_perturbation_scenarios(G_scheduled,
    n_scenarios=1,
    entry_delay_mean=5*60,
    entry_delay_std=2*60,
    running_delay_mean=-1*60,
    running_delay_std=0.5*60,
    seed=42,
)

## Propagation

### Plotly

In [19]:
realized_graphs = []

for node_p, edge_p in zip(
    node_perturbations,
    edge_perturbations,
):

    G_realized = propagate2(
        G_scheduled,
        edge_perturbations=edge_p,
    )

    realized_graphs.append(G_realized)


fig, ax = plot_ean_plotly(
    G_scheduled,
    nodesDf,
    edgesDf,
    title="Scheduled vs realized",
)


for G_realized in realized_graphs:

    draw_ean_plotly(
        G_realized,
        nodesDf,
        ax,
        alpha=0.5,
        linewidth_scale=0.8,
    )


show_ean(
    fig,
    filename=f"ean_visualization{scenario}.html",
    auto_open=True,
)

EAN visualization written to: C:\Users\LeoC\VSCodes\optimizationVinschgau\AusbauVinschgau\ean_simulation\ean_visualization1a.html


WindowsPath('ean_visualization1a.html')

### Matplotlib

In [15]:
'''realized_graphs = []

for node_p, edge_p in zip(node_perturbations, edge_perturbations):

    G_realized = propagate2(
        G_scheduled,
        edge_perturbations=edge_p,
    )

    realized_graphs.append(G_realized)


fig, ax = plot_ean(G_scheduled,nodesDf,edgesDf,title="Scheduled vs realized")

for G_realized in realized_graphs:
    draw_ean(G_realized,nodesDf,ax,alpha=0.2,linewidth_scale=0.8,)'''

'realized_graphs = []\n\nfor node_p, edge_p in zip(node_perturbations, edge_perturbations):\n\n    G_realized = propagate2(\n        G_scheduled,\n        edge_perturbations=edge_p,\n    )\n\n    realized_graphs.append(G_realized)\n\n\nfig, ax = plot_ean(G_scheduled,nodesDf,edgesDf,title="Scheduled vs realized")\n\nfor G_realized in realized_graphs:\n    draw_ean(G_realized,nodesDf,ax,alpha=0.2,linewidth_scale=0.8,)'

## Report

In [16]:
def format_time(seconds):
    seconds = round(seconds)
    sign = "-" if seconds < 0 else ""
    seconds = abs(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{sign}{hours:02d}:{minutes:02d}:{seconds:02d}"


def get_train_times(G):
    """Returns {train_id: {"dep": ..., "arr": ...}}."""
    times = {}
    trains = sorted({data["train"] for _, data in G.nodes(data=True) if "train" in data})

    for train in trains:
        events = [data for _, data in G.nodes(data=True) if data.get("train") == train]
        times[train] = {
            "dep": min(e["time"] for e in events if e["event"] == "dep"),
            "arr": max(e["time"] for e in events if e["event"] == "arr")
        }

    return times


def compare_to_schedule(G_scheduled, G_realized):
    scheduled = get_train_times(G_scheduled)
    realized = get_train_times(G_realized)

    report = {}

    for train in scheduled:
        report[train] = {
            "dep_sched": scheduled[train]["dep"],
            "dep_real": realized[train]["dep"],
            "arr_sched": scheduled[train]["arr"],
            "arr_real": realized[train]["arr"],
            "delay": realized[train]["arr"] - scheduled[train]["arr"]
        }

    return report


for i, (node_perturbation, edge_perturbation, G_realized) in enumerate(
    zip(node_perturbations, edge_perturbations, realized_graphs)
):
    arrival_report = compare_to_schedule(G_scheduled, G_realized)

    print(f"\nScenario {i}")
    print("-" * 100)

    for train in sorted(arrival_report):
        r = arrival_report[train]

        print(
            f"Train {train:>2}: "
            f"dep_sched={format_time(r['dep_sched'])}   "
            f"dep_real={format_time(r['dep_real'])}   "
            f"arr_sched={format_time(r['arr_sched'])}   "
            f"arr_real={format_time(r['arr_real'])}   "
            f"delay={format_time(r['delay'])}"
        )


Scenario 0
----------------------------------------------------------------------------------------------------
Train  1: dep_sched=10:31:18   dep_real=10:31:18   arr_sched=11:53:12   arr_real=11:53:18   delay=00:00:06
Train  2: dep_sched=11:27:18   dep_real=11:27:18   arr_sched=12:27:28   arr_real=12:27:28   delay=00:00:00
Train  3: dep_sched=11:31:18   dep_real=11:31:18   arr_sched=12:53:12   arr_real=12:53:18   delay=00:00:06
Train  4: dep_sched=11:51:18   dep_real=11:51:18   arr_sched=13:24:14   arr_real=13:25:12   delay=00:00:58
Train  5: dep_sched=12:27:18   dep_real=12:27:18   arr_sched=13:27:28   arr_real=13:28:58   delay=00:01:30
Train  6: dep_sched=12:31:18   dep_real=12:31:18   arr_sched=13:53:12   arr_real=13:54:46   delay=00:01:34
Train  7: dep_sched=12:51:18   dep_real=12:51:18   arr_sched=14:24:14   arr_real=14:25:18   delay=00:01:04
Train  8: dep_sched=13:27:18   dep_real=13:27:18   arr_sched=14:27:28   arr_real=14:29:03   delay=00:01:35
Train  9: dep_sched=13:31:18   

In [17]:
stop

NameError: name 'stop' is not defined

# Manual Perturbation

## Propagation

In [ ]:
#inject delays manually
running_edges = [(u, v) for u, v, data in G_scheduled.edges(data=True) if data["kind"] == "running"]
edge = running_edges[0]  

#perturbations = [{},{edge: 60},{edge: 120},{edge: 300}]
perturbations = [{edge:0}]


realized_graphs = []

for p in perturbations:

    G_realized = propagate(G_scheduled, p)

    realized_graphs.append(G_realized)


fig, ax = plot_ean(G_scheduled, nodesDf, edgesDf, title="Scheduled vs realized")

for G_realized in realized_graphs:
    draw_ean(G_realized,nodesDf,ax,alpha=0.2,linewidth_scale=0.8,)

## Report

In [ ]:
for i, (node_perturbation, edge_perturbation, G_realized) in enumerate(
    zip(node_perturbations, edge_perturbations, realized_graphs)
):
    arrival_report = compare_to_schedule(G_scheduled, G_realized)

    print(f"\nScenario {i}")
    print("-" * 100)

    for train in sorted(arrival_report):
        r = arrival_report[train]

        print(
            f"Train {train:>2}: "
            f"dep_sched={format_time(r['dep_sched'])}   "
            f"dep_real={format_time(r['dep_real'])}   "
            f"arr_sched={format_time(r['arr_sched'])}   "
            f"arr_real={format_time(r['arr_real'])}   "
            f"delay={format_time(r['delay'])}"
        )